# Benchmark Grover's Search Algorithm

Grover's algorithm is a foundational quantum algorithm, designed to solve the **unstructured search problem**: finding one or more "marked" items within a set of $N$ possibilities when no structure can be exploited. Classically, solving this problem requires $O(N)$ queries to an oracle that checks whether a candidate is a solution. Grover's algorithm achieves a quadratic speedup, requiring only $O(\sqrt{N})$ oracle queries.

***

### The model

We implements **Grover's search** in its minimal form, we perform one call to the Grover operator (no repeated iterations), where the oracle marks a single solution.

The circuit prepares a uniform superposition over $n$ qubits, applies the Grover operator once (oracle + diffuser), then measures. With one iteration, the marked states are amplified but not necessarily dominant; this is useful for understanding the algorithm or when the number of solutions is large relative to the search space.

We take a simple predicate function for marking the good states: 
$$
f(x)= (x==C),
$$
for $x$ a quantum number of $n$ qubits, $x=0,1,\dots 2^n-1$, and C being some integer.

### The Scoring function
The score is defined in terms of the success propabability of the marked states. For an initial state,
$$|\psi\rangle = \sum_a a_x |x\rangle,$$
if there are $M$ marked states, the probability to obtain one of the marked states is given by  
$$P_{\text{good}} = \sum_{x\in M} |a_x|^2~.$$
The score is defined as $S=P_{\text{good}} $, which obtains values between zero and one (the perfect score).

***
***


## Defining a `BenchmarkExample` for Grover

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES

from classiq import *

In [2]:
# We mark a single solution
MARKED_STATE = 5


# Define the number of grover iterations for a single marked state
def num_grover_iterations(num_qubits):
    N = 2**num_qubits
    theta = np.arcsin(1 / np.sqrt(N))
    r = np.floor(np.pi / (4 * theta) - 0.5)
    P_success = np.sin((2 * r + 1) * theta) ** 2
    return r, P_success

### Oracle and main circuit

The oracle flips the phase of any basis state whose index is in `MARKED_STATE`. We build a predicate that is 1 when the register (interpreted as an integer in big-endian order) is in that set, and 0 otherwise.

In [3]:
GROVER_DESCRIPTION = Path("../descriptions/grover.tex").read_text(encoding="utf-8")

In [4]:
class GroverExample(BenchmarkExample):
    def __init__(self, num_qubits: int, c: int):
        super().__init__(
            name="grover",
            num_qubits=num_qubits,
            report_family_title="Grover Search Algorithm",
            report_family_description=GROVER_DESCRIPTION,
        )
        assert c <= 2**num_qubits
        self.c = c

    def create_main(self) -> callable:
        @qperm
        def marked_oracle(x: Const[QNum], res: QBit) -> None:
            res ^= x == self.c

        @qfunc
        def main(x: Output[QNum[self.num_qubits]]):
            allocate(x)
            hadamard_transform(x)

            # Evaluate the number of grover iterations
            r, _ = num_grover_iterations(self.num_qubits)

            power(
                r,
                lambda: grover_operator(
                    lambda vars: phase_oracle(marked_oracle, vars),
                    hadamard_transform,
                    x,
                ),
            )

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_sample()
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df = result[0].value.dataframe

        marked_prob_sum = df.loc[df["x"] == MARKED_STATE, "probability"].sum()
        print(f"marked_prob_sum: {marked_prob_sum}")

        _, P_success = num_grover_iterations(self.num_qubits)

        score = marked_prob_sum / P_success
        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": float(score),
            "execution_time": exec_minutes,
        }

***
***
## Benchmarking a 4-qubits Grover

Define a specific example on 4 qubits:

In [5]:
example = GroverExample(num_qubits=4, c=5)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3BP18zKUsBoiwEYmI6O66dcPBgD


Define the number of shots and other hyperparameters:

In [6]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

### Choosing on which backend to run 

Define `HardwareRunner` for each backend. Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited after defining the `ResultCollector`.)*

In [7]:
classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:2]
]

ionq_runners = [
    HardwareRunner("IonQ", backend_name, **common_config)
    for backend_name in HARDWARES["IonQ"][0:1]
]


ibm_runners = [
    HardwareRunner(
        "IBM Quantum",
        backend_name,
        **common_config,
        backend_kwargs={
            "access_token": "PUT YOUR TOKEN HERE",
            "channel": "PUT YOUR CHANNEL HERE",
            "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
        },
    )
    for backend_name in HARDWARES["IBM Quantum"][0:1]
]

runners = [
    *classiq_runners,
    # *ionq_runners,
    # *ibm_runners,
]

In [8]:
print("Running for Backends:")
print(
    *[(runner.backend_service_provider, runner.backend_name) for runner in runners],
    sep="\n"
)

Running for Backends:
('Classiq', 'simulator')
('Classiq', 'simulator_statevector')


### Set a `ResultCollector` with a file name fixed for the current example

In [9]:
FILENAME = example.default_results_filename

In [10]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [11]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  grover-4 (2026-03-24 20:08:57.655640)   ==========
2026-03-24 20:09:02.696040: Submit grover-4 for Classiq - simulator_statevector
marked_prob_sum: 0.9084472656249968
2026-03-24 20:09:04.284583: Completed grover-4 for Classiq - simulator_statevector --> Score {'score': 0.9999999999999964, 'execution_time': 0.014900783333333334}
** Report updated: grover-4 for Classiq - simulator_statevector


In [12]:
await collector.print_status()

========== (2026-03-24 20:09:05.418339)   ==========
grover-4 | IonQ - qpu.forte-1 | COMPLETED | score=0.6241 | time=10.25 min
grover-4 | IBM Quantum - ibm_kingston | COMPLETED | score=0.3082 | time=50.94 min
grover-4 | Classiq - simulator | COMPLETED | score=0.9984 | time=0.01 min
grover-4 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min
